In [1]:
# 6.0.  (Hugging Face datasets)
!pip install -q datasets

import os
import json
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

print("from Hugging Face download VQA-RAD ...")
dataset = load_dataset("flaviagiammarino/vqa-rad")

base_dir = '/content/data'
images_dir = os.path.join(base_dir, 'images')
questions_file = os.path.join(base_dir, 'questions.json')
answers_file = os.path.join(base_dir, 'answers.json')


os.makedirs(images_dir, exist_ok=True)


all_questions = []
all_answers = []
processed_images = set()

print("save...")


splits = ['train', 'test']

for split in splits:
    print(f"处理 {split} 集...")
    for idx, item in enumerate(tqdm(dataset[split])):
        # Hugging Face  : {'image': PIL.Image, 'question': str, 'answer': str, ...}


        image_id = f"synpic_{split}_{idx}"


        img_path = os.path.join(images_dir, f"{image_id}.jpg")
        if image_id not in processed_images:
            item['image'].save(img_path)
            processed_images.add(image_id)

        q_dict = {
            "question_id": f"{split}_{idx}",
            "image_id": image_id,
            "question": item['question'],
            "split": split
        }
        all_questions.append(q_dict)


        a_dict = {
            "question_id": f"{split}_{idx}",
            "answer": str(item['answer'])
        }
        all_answers.append(a_dict)


with open(questions_file, 'w') as f:
    json.dump(all_questions, f)

with open(answers_file, 'w') as f:
    json.dump(all_answers, f)

print(f"\n✅ ok：")
print(f"- 图片文件夹: {images_dir} (共 {len(processed_images)} 张)")
print(f"- 问题文件: {questions_file} (共 {len(all_questions)} 条)")
print(f"- 答案文件: {answers_file} (共 {len(all_answers)} 条)")
print("\n run Dataset code！")

from Hugging Face download VQA-RAD ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-eb8844602202be(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e5bc3d208bb4dee(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

...

save...
处理 train 集...


100%|██████████| 1793/1793 [00:10<00:00, 177.05it/s]


处理 test 集...


100%|██████████| 451/451 [00:02<00:00, 183.63it/s]


✅ ok：
- 图片文件夹: /content/data/images (共 2244 张)
- 问题文件: /content/data/questions.json (共 2244 条)
- 答案文件: /content/data/answers.json (共 2244 条)

 run Dataset code！


In [4]:
# # ### 6.1 Data Loading and Preprocessing
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import json
import os

# ==========================================
# 1. Define the Dataset Class
# ==========================================
class VQARadDataset(Dataset):
    """VQA-RAD Dataset loader"""

    def __init__(self, image_dir, questions_file, answers_file, split='train'):
        self.image_dir = image_dir
        self.split = split

        # Load annotations
        print(f"Loading questions from: {questions_file}")
        with open(questions_file, 'r') as f:
            self.questions = json.load(f)

        print(f"Loading answers from: {answers_file}")
        with open(answers_file, 'r') as f:
            self.answers = json.load(f)

        # Build vocabularies
        self.build_vocabularies()

        # Image preprocessing
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def build_vocabularies(self):
        """Build question and answer vocabularies"""
        self.question_vocab = {'<pad>': 0, '<unk>': 1}
        self.answer_vocab = {}

        # Build Question vocabulary
        for q_data in self.questions:
            # Simple tokenization: lower case and split by space
            tokens = q_data['question'].lower().replace('?', '').split()
            for token in tokens:
                if token not in self.question_vocab:
                    self.question_vocab[token] = len(self.question_vocab)

        # Build Answer vocabulary
        for a_data in self.answers:
            answer = a_data['answer'].lower()
            if answer not in self.answer_vocab:
                self.answer_vocab[answer] = len(self.answer_vocab)

        print(f"Vocab built. Question tokens: {len(self.question_vocab)}, Answers: {len(self.answer_vocab)}")

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        # 1. Load Image
        # Note: Ensure the 'image_id' in json matches the filename in the directory
        img_name = self.questions[idx]['image_id']
        if not str(img_name).endswith('.jpg'):
            img_name = f"{img_name}.jpg"

        img_path = os.path.join(self.image_dir, img_name)

        # Safety check for image existence
        if not os.path.exists(img_path):
            # Return a dummy black image if file is missing (prevents crash during debug)
            image = Image.new('RGB', (224, 224))
        else:
            image = Image.open(img_path).convert('RGB')

        image = self.transform(image)

        # 2. Process Question
        question_text = self.questions[idx]['question'].lower().replace('?', '').split()
        question_ids = [self.question_vocab.get(q, self.question_vocab['<unk>'])
                       for q in question_text]

        # Pad or truncate to max length of 25
        max_len = 25
        question_ids = question_ids[:max_len]
        question_ids += [self.question_vocab['<pad>']] * (max_len - len(question_ids))

        # 3. Get Answer
        answer_text = self.answers[idx]['answer'].lower()
        answer_id = self.answer_vocab.get(answer_text, 0)

        return {
            'image': image,
            'question': torch.tensor(question_ids, dtype=torch.long),
            'answer': torch.tensor(answer_id, dtype=torch.long)
        }

# ==========================================
# 2. Instantiate and Test
# ==========================================

# Define paths (Using the /content/data structure from the previous step)
# NOTE: Ensure you have run the "Hugging Face Download Script" successfully before this.
data_root = '/content/data'

dataset = VQARadDataset(
    image_dir=os.path.join(data_root, 'images'),
    questions_file=os.path.join(data_root, 'questions.json'),
    answers_file=os.path.join(data_root, 'answers.json'),
    split='train'
)

# Initialize DataLoader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

# Verify loading
print("\n--- Testing Data Loading ---")
try:
    sample = next(iter(dataloader))
    print(f"🎉 Successfully loaded real data!")
    print(f"Image Batch Shape: {sample['image'].shape}")
    print(f"Question Batch Shape: {sample['question'].shape}")
    print(f"Answer Batch Shape: {sample['answer'].shape}")
except Exception as e:
    print(f"❌ Error loading batch: {e}")
    print("Tip: Make sure the 'data' folder exists and is populated.")

Loading questions from: /content/data/questions.json
Loading answers from: /content/data/answers.json
Vocab built. Question tokens: 1237, Answers: 517

--- Testing Data Loading ---
🎉 Successfully loaded real data!
Image Batch Shape: torch.Size([16, 3, 224, 224])
Question Batch Shape: torch.Size([16, 25])
Answer Batch Shape: torch.Size([16])


In [16]:
# ### 6.2 CNN Model Implementation
import torch
import torch.nn as nn
import torchvision.models as models


class CNNBaseline(nn.Module):
    """CNN-based VQA Model (Fixed Dimensions)"""

    def __init__(self, num_answers, question_vocab_size, embedding_dim=300):
        super(CNNBaseline, self).__init__()

        # Image encoder: ResNet-50
        self.image_encoder = models.resnet50(pretrained=True)
        self.image_encoder.fc = nn.Linear(2048, 1024)

        # Question encoder: LSTM
        self.embedding = nn.Embedding(question_vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=512,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        # Fusion module
        # 🔧 修正点：输入维度改为 1024 (Image) + 2048 (Question) = 3072
        self.fusion = nn.Sequential(
            nn.Linear(1024 + 2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        # Answer decoder
        self.decoder = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_answers)
        )

    def forward(self, image, question):
        # Image features
        image_features = self.image_encoder(image)  # [batch, 1024]

        # Question features
        embedded_question = self.embedding(question)
        _, (hidden, _) = self.lstm(embedded_question)
        # hidden shape: [4, batch, 512] (2 layers * 2 directions)

        question_features = hidden.permute(1, 0, 2).contiguous()
        question_features = question_features.view(question_features.size(0), -1)
        # result: [batch, 2048]

        # Fusion
        fused = torch.cat([image_features, question_features], dim=1) # [batch, 3072]
        fused = self.fusion(fused)

        # Answer prediction
        output = self.decoder(fused)
        return output


num_answers = 657

vocab_size = len(dataset.question_vocab) if 'dataset' in locals() else 8247

model = CNNBaseline(num_answers=num_answers, question_vocab_size=vocab_size)
model.to('cuda')
print("✅ 模型已修正并重新加载到 CUDA")

✅ 模型已修正并重新加载到 CUDA


In [6]:
### 6.3 Vision Transformer Model Implementation
from transformers import ViTFeatureExtractor, AutoModel
import torch
import torch.nn as nn

class ViTVQA(nn.Module):
    """Vision Transformer-based VQA Model"""

    def __init__(self, num_answers, device='cuda'):
        super(ViTVQA, self).__init__()
        self.device = device

        # Image encoder: ViT
        self.vit = AutoModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.image_projection = nn.Linear(768, 768)

        # Question encoder: BERT
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.question_projection = nn.Linear(768, 768)

        # Cross-modal attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=768,
            nhead=12,
            dim_feedforward=2048,
            dropout=0.1,
            batch_first=True
        )
        self.cross_attention = nn.TransformerEncoder(
            encoder_layer,
            num_layers=6
        )

        # Answer decoder
        self.decoder = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_answers)
        )

    def forward(self, image, question_ids, attention_mask):
        # Image features from ViT
        image_features = self.vit(image, interpolate_pos_encoding=True)
        image_cls = image_features.last_hidden_state[:, 0]  # [batch, 768]
        image_cls = self.image_projection(image_cls)

        # Question features from BERT
        question_features = self.bert(
            question_ids,
            attention_mask=attention_mask
        )
        question_cls = question_features.last_hidden_state[:, 0]  # [batch, 768]
        question_cls = self.question_projection(question_cls)

        # Combine for cross-attention
        combined = torch.cat([
            image_cls.unsqueeze(1),
            question_cls.unsqueeze(1)
        ], dim=1)  # [batch, 2, 768]

        # Cross-modal attention
        attended = self.cross_attention(combined)  # [batch, 2, 768]
        fused = attended.mean(dim=1)  # [batch, 768]

        # Answer prediction
        output = self.decoder(fused)  # [batch, num_answers]
        return output

# Instantiate model
model = ViTVQA(num_answers=657)
model.to('cuda')

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ViTVQA(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act_fn): GELUA

In [12]:
# 6.35 补丁代码
import torch
from torch.utils.data import DataLoader, random_split


data_root = '/content/data'
full_dataset = VQARadDataset(
    image_dir=os.path.join(data_root, 'images'),
    questions_file=os.path.join(data_root, 'questions.json'),
    answers_file=os.path.join(data_root, 'answers.json')
)


train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size


generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"✅ done: train {len(train_loader)} , val {len(val_loader)} ")


def calculate_class_weights():

    return None



Loading questions from: /content/data/questions.json
Loading answers from: /content/data/answers.json
Vocab built. Question tokens: 1237, Answers: 517
✅ done: train 113 , val 29 


In [17]:
### 6.4 Training Loop
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in dataloader:
        image = batch['image'].to(device)
        question = batch['question'].to(device)
        answer = batch['answer'].to(device)

        # Forward pass
        output = model(image, question)
        loss = criterion(output, answer)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Metrics
        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct += (predicted == answer).sum().item()
        total += answer.size(0)

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return accuracy, avg_loss

def train_model(model, train_loader, val_loader, num_epochs=50, device='cuda'):
    criterion = nn.CrossEntropyLoss(weight=calculate_class_weights())
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_acc = 0
    patience = 10
    patience_counter = 0

    for epoch in range(num_epochs):
        # Train
        train_acc, train_loss = train_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validate
        val_acc, val_loss = validate(model, val_loader, criterion, device)

        # Learning rate scheduling
        scheduler.step()

        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%")

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(torch.load('best_model.pt'))
    return model

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            image = batch['image'].to(device)
            question = batch['question'].to(device)
            answer = batch['answer'].to(device)

            output = model(image, question)
            loss = criterion(output, answer)

            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == answer).sum().item()
            total += answer.size(0)

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return accuracy, avg_loss

# Training
best_model = train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=50,
    device='cuda'
)


Epoch 1/50: Train Acc=23.12%, Val Acc=31.63%
Epoch 2/50: Train Acc=34.32%, Val Acc=33.41%
Epoch 3/50: Train Acc=37.72%, Val Acc=34.08%
Epoch 4/50: Train Acc=38.77%, Val Acc=36.08%
Epoch 5/50: Train Acc=41.17%, Val Acc=39.64%
Epoch 6/50: Train Acc=42.45%, Val Acc=39.64%
Epoch 7/50: Train Acc=46.41%, Val Acc=38.53%
Epoch 8/50: Train Acc=46.57%, Val Acc=40.53%
Epoch 9/50: Train Acc=48.91%, Val Acc=38.31%
Epoch 10/50: Train Acc=50.19%, Val Acc=41.43%
Epoch 11/50: Train Acc=50.31%, Val Acc=38.75%
Epoch 12/50: Train Acc=51.81%, Val Acc=42.32%
Epoch 13/50: Train Acc=52.31%, Val Acc=41.43%
Epoch 14/50: Train Acc=52.65%, Val Acc=41.43%
Epoch 15/50: Train Acc=54.37%, Val Acc=41.87%
Epoch 16/50: Train Acc=55.21%, Val Acc=42.54%
Epoch 17/50: Train Acc=55.88%, Val Acc=41.43%
Epoch 18/50: Train Acc=56.88%, Val Acc=42.54%
Epoch 19/50: Train Acc=57.05%, Val Acc=42.98%
Epoch 20/50: Train Acc=57.88%, Val Acc=43.88%
Epoch 21/50: Train Acc=58.83%, Val Acc=42.32%
Epoch 22/50: Train Acc=59.39%, Val Acc=41.8

In [1]:
# 6.5

# --------------------
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import json
import os
from transformers import BertTokenizer

class VQARadDatasetViT(Dataset):
    def __init__(self, image_dir, questions_file, answers_file, split='train'):
        self.image_dir = image_dir

        with open(questions_file, 'r') as f:
            self.questions = json.load(f)
        with open(answers_file, 'r') as f:
            self.answers = json.load(f)


        print("loading BERT Tokenizer ...")
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


        self.answer_vocab = {}
        for a_data in self.answers:
            answer = str(a_data['answer']).lower()
            if answer not in self.answer_vocab:
                self.answer_vocab[answer] = len(self.answer_vocab)


        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):

        img_name = self.questions[idx]['image_id']
        if not str(img_name).endswith('.jpg'):
            img_name = f"{img_name}.jpg"
        img_path = os.path.join(self.image_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))
        image = self.transform(image)


        question_text = self.questions[idx]['question']
        encoding = self.tokenizer(
            question_text,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=32,
            add_special_tokens=True
        )


        answer_text = str(self.answers[idx]['answer']).lower()
        answer_id = self.answer_vocab.get(answer_text, 0)

        return {
            'image': image,
            'question_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'answer': torch.tensor(answer_id, dtype=torch.long)
        }

print("✅ finish")



✅ finish


In [2]:
# 6.6 ViT


data_root = '/content/data'


full_dataset = VQARadDatasetViT(
    image_dir=os.path.join(data_root, 'images'),
    questions_file=os.path.join(data_root, 'questions.json'),
    answers_file=os.path.join(data_root, 'answers.json')
)


train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)



num_classes = len(full_dataset.answer_vocab)
print(f"分类数量: {num_classes}")

model = ViTVQA(num_answers=num_classes)
model.to('cuda')

print("finish2")

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/questions.json'

In [3]:
import os

print("--- Checking Directory Structure ---")

# Check if /content/data exists
if os.path.exists('/content/data'):
    print("Found /content/data folder.")
    print("Files inside /content/data:")
    print(os.listdir('/content/data'))
else:
    print("ERROR: /content/data folder does NOT exist.")
    print("Files in /content root:")
    print(os.listdir('/content'))

print("--- End Check ---")

--- Checking Directory Structure ---
ERROR: /content/data folder does NOT exist.
Files in /content root:
['.config', 'sample_data']
--- End Check ---


In [4]:
# Re-download and prepare data
# ---------------------------------------------------------
!pip install -q datasets

import os
import json
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

print("Downloading dataset from Hugging Face...")
dataset = load_dataset("flaviagiammarino/vqa-rad")

base_dir = '/content/data'
images_dir = os.path.join(base_dir, 'images')
questions_file = os.path.join(base_dir, 'questions.json')
answers_file = os.path.join(base_dir, 'answers.json')

os.makedirs(images_dir, exist_ok=True)

all_questions = []
all_answers = []
processed_images = set()

print("Processing and saving images...")
splits = ['train', 'test']

for split in splits:
    for idx, item in enumerate(dataset[split]):
        image_id = f"synpic_{split}_{idx}"

        # Save Image
        img_path = os.path.join(images_dir, f"{image_id}.jpg")
        if image_id not in processed_images:
            item['image'].save(img_path)
            processed_images.add(image_id)

        # Save Question
        q_dict = {
            "question_id": f"{split}_{idx}",
            "image_id": image_id,
            "question": item['question'],
            "split": split
        }
        all_questions.append(q_dict)

        # Save Answer
        a_dict = {
            "question_id": f"{split}_{idx}",
            "answer": str(item['answer'])
        }
        all_answers.append(a_dict)

with open(questions_file, 'w') as f:
    json.dump(all_questions, f)

with open(answers_file, 'w') as f:
    json.dump(all_answers, f)

print("Data preparation complete.")
print(f"Images saved: {len(processed_images)}")
print(f"Questions saved: {len(all_questions)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-eb8844602202be(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e5bc3d208bb4dee(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/451 [00:00<?, ? examples/s]

Processing and saving images...
Data preparation complete.
Images saved: 2244
Questions saved: 2244


In [5]:
# 6.7

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import json
import os
from transformers import BertTokenizer, ViTFeatureExtractor, AutoModel

# ---------------------------------------------------------
# 1. Define Dataset Class for ViT (Data Processor)
# ---------------------------------------------------------
class VQARadDatasetViT(Dataset):
    def __init__(self, image_dir, questions_file, answers_file, split='train'):
        self.image_dir = image_dir
        with open(questions_file, 'r') as f:
            self.questions = json.load(f)
        with open(answers_file, 'r') as f:
            self.answers = json.load(f)

        print("Loading BERT Tokenizer...")
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

        self.answer_vocab = {}
        for a_data in self.answers:
            answer = str(a_data['answer']).lower()
            if answer not in self.answer_vocab:
                self.answer_vocab[answer] = len(self.answer_vocab)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        img_name = self.questions[idx]['image_id']
        if not str(img_name).endswith('.jpg'):
            img_name = f"{img_name}.jpg"
        img_path = os.path.join(self.image_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))
        image = self.transform(image)

        question_text = self.questions[idx]['question']
        encoding = self.tokenizer(
            question_text,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=32,
            add_special_tokens=True
        )

        answer_text = str(self.answers[idx]['answer']).lower()
        answer_id = self.answer_vocab.get(answer_text, 0)

        return {
            'image': image,
            'question_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'answer': torch.tensor(answer_id, dtype=torch.long)
        }

# ---------------------------------------------------------
# 2. Define ViT Model Architecture (Model Structure)
# ---------------------------------------------------------
class ViTVQA(nn.Module):
    def __init__(self, num_answers, device='cuda'):
        super(ViTVQA, self).__init__()
        self.device = device

        # Image encoder: ViT
        self.vit = AutoModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.image_projection = nn.Linear(768, 768)

        # Question encoder: BERT
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.question_projection = nn.Linear(768, 768)

        # Cross-modal attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=768, nhead=12, dim_feedforward=2048, dropout=0.1, batch_first=True
        )
        self.cross_attention = nn.TransformerEncoder(encoder_layer, num_layers=6)

        # Answer decoder
        self.decoder = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_answers)
        )

    def forward(self, image, question_ids, attention_mask):
        image_features = self.vit(image, interpolate_pos_encoding=True)
        image_cls = image_features.last_hidden_state[:, 0]
        image_cls = self.image_projection(image_cls)

        question_features = self.bert(question_ids, attention_mask=attention_mask)
        question_cls = question_features.last_hidden_state[:, 0]
        question_cls = self.question_projection(question_cls)

        combined = torch.cat([image_cls.unsqueeze(1), question_cls.unsqueeze(1)], dim=1)
        attended = self.cross_attention(combined)
        fused = attended.mean(dim=1)

        output = self.decoder(fused)
        return output

# ---------------------------------------------------------
# 3. Instantiate and Load (Action!)
# ---------------------------------------------------------
print("Setting up data and model...")
data_root = '/content/data'

# Load Dataset
full_dataset = VQARadDatasetViT(
    image_dir=os.path.join(data_root, 'images'),
    questions_file=os.path.join(data_root, 'questions.json'),
    answers_file=os.path.join(data_root, 'answers.json')
)

# Split Data
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

# Create Loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Initialize Model
num_classes = len(full_dataset.answer_vocab)
print(f"Number of classes: {num_classes}")

model = ViTVQA(num_answers=num_classes)
model.to('cuda')

print("ViT Model is ready for training!")

Setting up data and model...
Loading BERT Tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Number of classes: 517


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ViT Model is ready for training!


In [7]:
# 6.8

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn

# ---------------------------------------------------------
# 1. Define Training & Validation Functions (ViT Special)
# ---------------------------------------------------------
def train_epoch_vit(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0; correct = 0; total = 0

    for batch in dataloader:
        # Move data to GPU
        image = batch['image'].to(device)
        question_ids = batch['question_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        answer = batch['answer'].to(device)

        # Forward pass (3 arguments for ViT!)
        output = model(image, question_ids, attention_mask)
        loss = criterion(output, answer)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Metrics
        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct += (predicted == answer).sum().item()
        total += answer.size(0)

    return 100 * correct / total, total_loss / len(dataloader)

def validate_vit(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in dataloader:
            image = batch['image'].to(device)
            question_ids = batch['question_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            answer = batch['answer'].to(device)

            output = model(image, question_ids, attention_mask)
            loss = criterion(output, answer)

            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == answer).sum().item()
            total += answer.size(0)
    return 100 * correct / total, total_loss / len(dataloader)

# ---------------------------------------------------------
# 2. Start the Training Loop!
# ---------------------------------------------------------
criterion = nn.CrossEntropyLoss()
# Using a smaller learning rate for ViT (2e-5) as per best practices
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=50)
best_val_acc = 0

print("🚀 Starting ViT Training (Target: ~76% Accuracy)...")
print("This may take 40-60 minutes. Please keep the tab open.")

for epoch in range(50):
    train_acc, train_loss = train_epoch_vit(model, train_loader, criterion, optimizer, 'cuda')
    val_acc, val_loss = validate_vit(model, val_loader, criterion, 'cuda')
    scheduler.step()

    print(f"Epoch {epoch+1}/50: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_vit_model.pt')

print(f"🏁 Training Finished! Best Validation Accuracy: {best_val_acc:.2f}%")

🚀 Starting ViT Training (Target: ~76% Accuracy)...
This may take 40-60 minutes. Please keep the tab open.
Epoch 1/50: Train Acc=27.19%, Val Acc=32.52%
Epoch 2/50: Train Acc=34.48%, Val Acc=35.41%
Epoch 3/50: Train Acc=39.33%, Val Acc=36.08%
Epoch 4/50: Train Acc=46.35%, Val Acc=39.20%
Epoch 5/50: Train Acc=49.92%, Val Acc=40.76%
Epoch 6/50: Train Acc=53.37%, Val Acc=40.53%
Epoch 7/50: Train Acc=55.26%, Val Acc=38.75%
Epoch 8/50: Train Acc=59.28%, Val Acc=41.87%
Epoch 9/50: Train Acc=62.01%, Val Acc=44.77%
Epoch 10/50: Train Acc=66.13%, Val Acc=43.43%
Epoch 11/50: Train Acc=69.14%, Val Acc=46.10%
Epoch 12/50: Train Acc=72.65%, Val Acc=47.66%
Epoch 13/50: Train Acc=75.71%, Val Acc=47.44%
Epoch 14/50: Train Acc=78.61%, Val Acc=46.99%
Epoch 15/50: Train Acc=80.89%, Val Acc=48.11%
Epoch 16/50: Train Acc=83.73%, Val Acc=49.44%
Epoch 17/50: Train Acc=85.18%, Val Acc=49.89%
Epoch 18/50: Train Acc=87.41%, Val Acc=49.22%
Epoch 19/50: Train Acc=89.36%, Val Acc=51.45%
Epoch 20/50: Train Acc=90.08%

In [11]:
# 6.9
import torch.optim as optim
import time


epochs = 50
learning_rate = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()


model = model.to(device)

print(f"🚀 Training started on {device}...")
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}


for epoch in range(epochs):
    start_time = time.time()
    model.train() #
    running_loss = 0.0
    correct = 0
    total = 0


    for batch_data in train_loader:

        batch_data = [item.to(device) for item in batch_data]


        images = batch_data[0]
        questions = batch_data[1]
        labels = batch_data[-1]



        optimizer.zero_grad()


        try:

            outputs = model(images, questions)
        except TypeError:

            masks = batch_data[2]
            outputs = model(images, questions, masks)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)


    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch_data in val_loader:
            batch_data = [item.to(device) for item in batch_data]
            images = batch_data[0]
            questions = batch_data[1]
            labels = batch_data[-1]

            try:
                outputs = model(images, questions)
            except TypeError:
                masks = batch_data[2]
                outputs = model(images, questions, masks)

            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    history['val_acc'].append(val_acc)


    end_time = time.time()
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {epoch_loss:.4f} | "
          f"Train Acc: {epoch_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}% | "
          f"Time: {end_time - start_time:.1f}s")

print("🏁 Training Finished!")

🚀 Training started on cuda...


AttributeError: 'str' object has no attribute 'to'